In [2]:
!pip install torch torchvision pandas numpy Pillow open_clip_torch tqdm scikit-learn

In [3]:
import os
import ast
import pandas as pd
import numpy as np
import torch
from PIL import Image
import open_clip
from tqdm import tqdm
from sklearn.model_selection import train_test_split

/data/rahul_mishra/miniconda3/envs/rfp_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
LOCAL_BASE_DIR = "/data/rahul_mishra/RFP" 

DATASET_FOLDERS = [
    "olx_bikes_dataset/",
    "olx_books_dataset/",
    "olx_cars_dataset/",
    "olx_cycle_dataset/",
    "olx_flat_dataset/",
    "olx_fridges_dataset/",
    "olx_games_dataset/",
    "olx_gamesentertainment_dataset/",
    "olx_laptop_dataset/",
    "olx_mobile_dataset/",
    "olx_phones_dataset/",
    "olx_printer_dataset/",
    "olx_tv_dataset/",
    "olx_washingmachine_dataset/"
]

In [7]:
missing_csv_files = []
missing_image_files = []

def get_all_local_image_paths(images_str):
    try:
        img_list = ast.literal_eval(images_str)
        if not isinstance(img_list, list) or len(img_list) == 0:
            return []

        valid_paths = []
        
        for raw_path in img_list:
            if "MyDrive/" in raw_path:
                relative_path = raw_path.split("MyDrive/")[-1].lstrip("/")
            else:
                relative_path = raw_path.lstrip("/")

            expected_path = os.path.join(LOCAL_BASE_DIR, relative_path)
            expected_path = os.path.normpath(expected_path)

            if os.path.exists(expected_path):
                valid_paths.append(expected_path)
            else:
                missing_image_files.append(expected_path)

        return valid_paths

    except Exception:
        return []

def load_unified_data():
    dfs = []

    for folder in DATASET_FOLDERS:
        folder_path = os.path.join(LOCAL_BASE_DIR, folder)
        csv_path = os.path.join(folder_path, "dataset.csv")

        if not os.path.exists(csv_path):
            missing_csv_files.append(csv_path)
            continue

        try:
            df = pd.read_csv(csv_path, encoding="utf-8")
        except UnicodeDecodeError:
            df = pd.read_csv(csv_path, encoding="latin1", encoding_errors="ignore")

        df = df.dropna(subset=["price", "title", "description", "images"])
        df["price"] = pd.to_numeric(df["price"], errors="coerce")
        df = df[df["price"] > 0].copy()
        
        df["valid_image_paths"] = df["images"].apply(get_all_local_image_paths)
        df = df[df["valid_image_paths"].map(len) > 0].copy()

        df["category"] = folder.replace("olx_", "").replace("_dataset", "")
        dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    unified_df = pd.concat(dfs, ignore_index=True)
    unified_df["log_price"] = np.log(unified_df["price"].astype(float))
    unified_df = unified_df.reset_index(drop=True)

    return unified_df

df = load_unified_data()

# print("="*50)
# print("MISSING FILES REPORT")
# print("="*50)

# print(f"\nMissing dataset.csv files: {len(missing_csv_files)}")
# for csv in missing_csv_files:
#     print(f" - {csv}")

# print(f"\nMissing Image files: {len(missing_image_files)}")
# for img in missing_image_files[:50]:
#     print(f" - {img}")

# if len(missing_image_files) > 50:
#     print(f" ... and {len(missing_image_files) - 50} more images missing.")

print("\n" + "-"*50)
print(f"Total Unified Samples Ready: {len(df)}")


--------------------------------------------------
Total Unified Samples Ready: 4998


In [6]:
df.head()

,id,url,title,description,price,images,image_count,scraped_at,valid_image_paths,category,log_price
0,car_7d84be138c,https://www.olx.in/item/motorcycles-c81-used-b...,bajaj platina (2008),Motor bike - Motorcycles,15000,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,43:19.5,[/data/rahul_mishra/RFP/olx_bikes_dataset/imag...,bikes/,9.615805
1,car_b9d6dc8a84,https://www.olx.in/item/motorcycles-c81-used-t...,tvs star city plus (2025),Get Exciting Offers!! Star City Plus – Style T...,87805,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,43:30.8,[/data/rahul_mishra/RFP/olx_bikes_dataset/imag...,bikes/,11.382874
2,car_a9eaed64a0,https://www.olx.in/item/motorcycles-c81-used-b...,bajaj pulsar n160 (2024),Selling my Bajaj Pulsar N160 which is almost l...,145000,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,43:51.4,[/data/rahul_mishra/RFP/olx_bikes_dataset/imag...,bikes/,11.884489
3,car_92ec39b2d9,https://www.olx.in/item/motorcycles-c81-used-h...,honda cb350 (2025),Get Exciting Offers!! Honda CB350 - The Classi...,256577,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,44:15.3,[/data/rahul_mishra/RFP/olx_bikes_dataset/imag...,bikes/,12.455184
4,car_bc83aeb8e7,https://www.olx.in/item/motorcycles-c81-used-h...,hero xoom 160 (2020),Urgent money.. I'm goning usa - Motorcycles,52000,"[""/content/drive/MyDrive/olx_bikes_dataset/ima...",18,45:17.1,[/data/rahul_mishra/RFP/olx_bikes_dataset/imag...,bikes/,10.858999


In [20]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

print(f"Using device: {torch.cuda.get_device_name(0)}")

Using device: NVIDIA H100 PCIe


In [23]:
def generate_and_save_combined_embeddings(df, output_folder="category_combined_embeddings"):
    os.makedirs(output_folder, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='laion2b_s32b_b82k')
    tokenizer = open_clip.get_tokenizer('ViT-L-14')
    model = model.to(device)
    model.eval()

    categories = df['category'].unique()

    for cat in categories:
        print(f"\nProcessing category: {cat}")
        cat_df = df[df['category'] == cat]
        
        cat_combined_ids = []
        cat_combined_embeds = []
        
        with torch.no_grad():
            for idx, row in tqdm(cat_df.iterrows(), total=len(cat_df)):
                
                # 1. Process Text (768 dimensions)
                text_data = str(row['description'])[:7000]
                text_feat = None
                try:
                    text_tokens = tokenizer([text_data]).to(device)
                    text_feat = model.encode_text(text_tokens)
                    text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
                except Exception:
                    pass

                # 2. Process Images (768 dimensions)
                valid_paths = row['valid_image_paths']
                img_tensors = []
                avg_img_feat = None
                for img_path in valid_paths:
                    try:
                        img = preprocess(Image.open(img_path).convert('RGB')).unsqueeze(0).to(device)
                        img_tensors.append(img)
                    except Exception:
                        continue
                
                if img_tensors:
                    batch_imgs = torch.cat(img_tensors)
                    img_feats = model.encode_image(batch_imgs)
                    img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
                    
                    avg_img_feat = img_feats.mean(dim=0, keepdim=True)
                    avg_img_feat = avg_img_feat / avg_img_feat.norm(dim=-1, keepdim=True)

                # 3. Combine into 1536 dimensions (ONLY if both exist)
                if text_feat is not None and avg_img_feat is not None:
                    # Concatenate the [1, 768] Image and [1, 768] Text into [1, 1536]
                    combined_feat = torch.cat([avg_img_feat, text_feat], dim=1)
                    
                    cat_combined_embeds.append(combined_feat.cpu())
                    cat_combined_ids.append(idx)

        safe_cat = str(cat).replace("/", "")

        if cat_combined_embeds:
            torch.save({
                'dataframe_indices': cat_combined_ids,
                'embeddings': torch.cat(cat_combined_embeds, dim=0),
                'log_prices': torch.tensor(cat_df.loc[cat_combined_ids, 'log_price'].values)
            }, os.path.join(output_folder, f"combined_{safe_cat}.pt"))

if 'df' in locals() and len(df) > 0:
    generate_and_save_combined_embeddings(df, output_folder="category_combined_embeddings")
else:
    print("Dataframe 'df' not found or is empty.")

Using device: cuda

Processing category: bikes/


100%|██████████| 289/289 [00:22<00:00, 12.75it/s]



Processing category: books/


100%|██████████| 302/302 [00:09<00:00, 31.05it/s]



Processing category: cars/


...

100%|██████████| 263/263 [00:08<00:00, 32.48it/s]



Processing category: tv/


100%|██████████| 480/480 [00:16<00:00, 28.75it/s]



Processing category: washingmachine/


100%|██████████| 473/473 [00:15<00:00, 31.34it/s]


In [24]:
def split_category_embeddings(df, input_folder="category_combined_embeddings", output_folder="split_embeddings"):
    os.makedirs(os.path.join(output_folder, 'train'), exist_ok=True)
    os.makedirs(os.path.join(output_folder, 'val'), exist_ok=True)
    os.makedirs(os.path.join(output_folder, 'test'), exist_ok=True)

    for file in os.listdir(input_folder):
        if not file.endswith('.pt'):
            continue
            
        file_path = os.path.join(input_folder, file)
        data = torch.load(file_path, weights_only=False)
        
        indices = data['dataframe_indices']
        embeddings = data['embeddings']

        prices = torch.tensor(df.loc[indices, 'log_price'].values, dtype=torch.float32)

        idx_train, idx_temp, emb_train, emb_temp, price_train, price_temp = train_test_split(
            indices, embeddings, prices, test_size=0.3, random_state=42
        )
        
        idx_val, idx_test, emb_val, emb_test, price_val, price_test = train_test_split(
            idx_temp, emb_temp, price_temp, test_size=0.5, random_state=42
        )

        torch.save({
            'dataframe_indices': idx_train,
            'embeddings': emb_train,
            'log_prices': price_train
        }, os.path.join(output_folder, 'train', file))
        
        torch.save({
            'dataframe_indices': idx_val,
            'embeddings': emb_val,
            'log_prices': price_val
        }, os.path.join(output_folder, 'val', file))
        
        torch.save({
            'dataframe_indices': idx_test,
            'embeddings': emb_test,
            'log_prices': price_test
        }, os.path.join(output_folder, 'test', file))

        print(f"{file} -> Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}")

if 'df' in locals():
    split_category_embeddings(df)

combined_bikes.pt -> Train: 202 | Val: 43 | Test: 44
combined_books.pt -> Train: 210 | Val: 45 | Test: 46
combined_cars.pt -> Train: 144 | Val: 31 | Test: 31
combined_cycle.pt -> Train: 133 | Val: 29 | Test: 29
combined_flat.pt -> Train: 235 | Val: 51 | Test: 51
combined_fridges.pt -> Train: 350 | Val: 75 | Test: 75
combined_games.pt -> Train: 7 | Val: 2 | Test: 2
combined_gamesentertainment.pt -> Train: 347 | Val: 75 | Test: 75
combined_laptop.pt -> Train: 584 | Val: 125 | Test: 126
combined_mobile.pt -> Train: 219 | Val: 47 | Test: 48
combined_phones.pt -> Train: 208 | Val: 45 | Test: 45
combined_printer.pt -> Train: 183 | Val: 39 | Test: 40
combined_tv.pt -> Train: 335 | Val: 72 | Test: 72
combined_washingmachine.pt -> Train: 331 | Val: 71 | Test: 71


In [ ]:
!pip install torch

In [4]:
import os
import torch

def merge_mobile_into_phones(split_dir="split_embeddings"):
    print("Merging 'MOBILE' data into 'PHONES' dataset...\n")
    
    for split in ['train', 'test', 'val']:
        folder_path = os.path.join(split_dir, split)
        if not os.path.exists(folder_path): 
            continue
        
        mobile_path = os.path.join(folder_path, "combined_mobile.pt")
        phones_path = os.path.join(folder_path, "combined_phones.pt")
        
        if os.path.exists(mobile_path) and os.path.exists(phones_path):
            # Load both datasets
            mobile_data = torch.load(mobile_path, weights_only=False)
            phones_data = torch.load(phones_path, weights_only=False)
            
            # Mathematically stack the embeddings and prices together
            merged_data = {
                'embeddings': torch.cat([phones_data['embeddings'], mobile_data['embeddings']], dim=0),
                'log_prices': torch.cat([phones_data['log_prices'], mobile_data['log_prices']], dim=0)
            }
            
            # Overwrite the existing PHONES file with the newly enriched data
            torch.save(merged_data, phones_path)
            
            # Delete the obsolete MOBILE file so it doesn't train twice
            os.remove(mobile_path)
            
            print(f"[{split.upper()}] Success: Merged MOBILE into PHONES.")
            print(f"    -> New PHONES dataset size: {len(merged_data['embeddings'])} items")
            
        elif os.path.exists(phones_path):
            print(f"[{split.upper()}] MOBILE file already deleted or missing. PHONES dataset remains unchanged.")
        else:
            print(f"[{split.upper()}] Missing necessary files to perform the merge.")

merge_mobile_into_phones()

Merging 'MOBILE' data into 'PHONES' dataset...

[TRAIN] MOBILE file already deleted or missing. PHONES dataset remains unchanged.
[TEST] MOBILE file already deleted or missing. PHONES dataset remains unchanged.
[VAL] MOBILE file already deleted or missing. PHONES dataset remains unchanged.
